# M4 — Exploratory Data Analysis

**Project question:** does weather carry information for a better hourly day-ahead demand forecast than EIA's own
forecast (DF), and how does demand depend on temperature in each region?

**Key number:** EIA's day-ahead forecast error — mean absolute percentage error (MAPE) of DF against actual demand D,
per region.

Rules for this notebook: run it with **Restart & Run All** on a fresh build (`python src/load.py`, then
`python src/clean.py`); the database is opened **read-only**; every query states its flag filter; every value reaches
SQL through a `?` placeholder; every figure is written by code to `figures/`.

Sections follow the M4 brief (2.1–2.8). Each section starts with the question it answers and its owner.

## 0. Parameters (every choice in one place)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else "src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from explore import connect_ro, query, table_hashes, expected_hours, to_local, save_fig

REGIONS = ["NYIS", "ERCO", "CISO"]
LOCAL_TZ = {"NYIS": "America/New_York", "ERCO": "America/Chicago", "CISO": "America/Los_Angeles"}
START, END = "2019-01-01", "2026-09-30"          # UTC dates covered by the landed data
EXPECTED_HOURS = expected_hours(START, END)     # per series

TRUSTED_FLAGS = ("ok",)                                     # main flag filter
TRUSTED_FLAGS_WIDE = ("ok", "unusual_high", "unusual_low")  # second filter: keep real extremes

DAILY_MIN_HOURS = 20    # a daily aggregate needs >= 20 of 24 hours (reason in the report, section 4)
MONTHLY_MIN_DAYS = 25   # a monthly aggregate needs >= 25 complete days

SEED = 42               # bootstrap random seed
N_BOOT = 2000           # bootstrap resamples
rng = np.random.default_rng(SEED)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

con = connect_ro()
HASHES_BEFORE = table_hashes(con)
print("expected hours per series:", EXPECTED_HOURS)
print("tables hashed:", len(HASHES_BEFORE))

## 2.1 Raw versus cleaned: the key number (Esma)

*Question:* how large is EIA's day-ahead forecast error per region, and how much does cleaning change that number?
Computed twice — on the M2 tables as loaded, and on the cleaned layer with the flag filter we report.

## 2.2 The question and the summary that answers it (Tuba)

*Question:* which summary answers "how wrong is the day-ahead forecast in a typical hour", and does an alternative
summary (median APE, 95th percentile) rank the regions the same way? Every value with its n.

## 2.3 Shape (Tuba)

*Question:* what does the distribution of forecast error (and demand) look like — skewed, heavy-tailed, bimodal — and
what does that mean for the summary chosen in 2.2? Histogram with a chosen bin width plus ECDF, overall and per region.

## 2.4 Coverage, denominators and the flag filter (Esma)

*Question:* what is each region's number made of? Coverage against the hours there could have been, the coverage rule
for every aggregate (with and without), and one key result under two flag filters.

## 2.5 Time (İrem)

*Question:* when in the day and the year is demand (and forecast error) high? Clock: **local time** per region for
human-behaviour patterns, converted explicitly with `tz_convert`. Edges, two time scales, the calendar (holidays).

## 2.6 Relationships (İrem)

*Question:* how does demand depend on temperature in each region? Plot first (binned medians), then Pearson and
Spearman; check whether one event (e.g. Winter Storm Uri, Feb 2021) or a shared seasonal pattern drives it.

## 2.7 One comparison, with an interval (Tuba)

*Question:* one comparison that matters for the project question, with a 95% bootstrap interval over **days**
(not hours), a composition check, and the number of comparisons we looked at before choosing it.

## 2.8 Findings (everyone)

The findings table, back-to-cleaning and forward-to-modelling lists live in `docs/eda_report.md`, section 8.
Every finding there points to a figure or a query output in this notebook.

## Decisions used and read-only check

In [ ]:
print("regions:", REGIONS)
print("local clocks:", LOCAL_TZ)
print("period:", START, "to", END, "| expected hours per series:", EXPECTED_HOURS)
print("flag filters:", TRUSTED_FLAGS, "| wide:", TRUSTED_FLAGS_WIDE)
print("coverage rules: daily >=", DAILY_MIN_HOURS, "h | monthly >=", MONTHLY_MIN_DAYS, "days")
print("bootstrap:", N_BOOT, "resamples, seed", SEED)

assert table_hashes(con) == HASHES_BEFORE, "the database changed while the notebook ran"
print("database unchanged: every table has the same hash as at the start")
con.close()